# 07 - Request -> approval -> publication with a policy template

## Learning objective
Demonstrate the decisions behind a maker submission, independent review, policy-template selection and controlled publication. Observe denied access before approval, narrowly permitted reads after approval, and a rejected update that leaves v1 available.

**Technical level:** L300. **Duration:** 25-35 minutes. **Prerequisites:** notebook 00, Python 3.12, `uv sync --frozen --all-extras --dev`, and the repository `.venv` kernel. **Cost and permissions:** none for this local lab; optional portal observation requires separately authorized read access, supported licensing and an existing synthetic example.

**This is LOCAL_SIMULATION_NOT_AGENT365_OR_ENTRA.** No cell submits a request, changes a template, accepts consent, publishes an agent, assigns an access package or changes Conditional Access. The supplied publication writer records DRY_RUN receipts only. The optional portal tour is manual and read-only.

Read the [participant lab](../docs/participant-lab/request-approval-publish.md) and [production/WAF disclaimer](../docs/participant-lab/README.md#production-and-well-architected-framework-waf-disclaimer). A local PASS is not a native integration claim.

## 1. Map the customer's two submission routes

A Copilot Studio maker requests organizational distribution through its Teams/Microsoft Copilot channel. A Foundry maker selects **People in your organization** when publishing to Microsoft 365 Copilot/Teams; **Just you** is a different route and does not require the same organizational approval. Native requests can appear under **Agents > All agents > Requests** as Pending review, Pending update or Pending activate. Inspect the actual channel, platform and tenant rollout rather than assuming identical UI behavior.

The administrative publication stages are **audience -> policy template -> permissions/admin consent -> publish**. They are separate from resource access and agent activation. This notebook does not build or publish a Copilot Studio/Foundry agent. The repository's function handlers execute in its Python caller; a portal publication would not automatically host them.

The code below imports the existing `run_lifecycle` implementation and the existing audit verifier. It does not import a cloud publisher or acquire a credential.

In [ ]:
import json
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

from foundry_security_accelerator.audit import verify_audit
from foundry_security_accelerator.lifecycle import run_lifecycle

assert sys.version_info[:2] == (3, 12), "Select the prepared Python 3.12 kernel"
print("LOCAL ONLY: no native request, consent, publication or policy change")

## 2. Run one isolated story, then inspect its evidence

`run_lifecycle(state_dir)` executes the **entire** synthetic story once: request v1, review seven checks, refuse maker self-approval, issue independent exact/expiring approval, create a DRY_RUN publication receipt, test access, review and reject an unsafe v2, and verify the audit. Later cells inspect that retained report; they do not pretend to perform incremental native operations.

A scoped temporary directory keeps this exercise isolated from other workshop runs. Re-running this cell creates a fresh story after cleaning the prior exercise directory. Nothing is sent to Azure. Equivalent CLI: `uv run python -m fsa --mode local lifecycle e2e`; the CLI retains its evidence under ignored local state instead of this notebook's temporary directory.

In [ ]:
if "workspace" in globals():
    globals()["workspace"].cleanup()
workspace = TemporaryDirectory(prefix="fsa-lifecycle-lab-")
report = run_lifecycle(Path(workspace.name))
assert report["kind"] == "LOCAL_SIMULATION_NOT_AGENT365_OR_ENTRA"
assert report["platform_actions"] == "NOT_EXECUTED"
assert report["status"] == "PASS"
print(
    json.dumps(
        {
            "evidence_kind": report["kind"],
            "local_request": report["request"]["status"],
            "publication": report["publication_receipt"]["status"],
            "native_actions": report["platform_actions"],
        },
        indent=2,
    )
)

## 3. Review before approval: seven evidence checks

REQ01 owner/sponsor; REQ02 distinct agent identity; REQ03 release gate; REQ04 write approval; REQ05 bounded outbound access; REQ06 clearance capped at Internal; REQ07 requested resources inside the bounded package. These are this application's checks, **not Microsoft Graph or Agent 365 API verdicts**.

Read the names and PASS/FAIL values from the actual report below. Identity strings and the `ai-admin` actor are synthetic. The lab enforces independent approval as an organizational teaching requirement; it does not prove that the native platform universally prevents every dual-role maker/admin from approving their own submission.

In [ ]:
initial_checks = report["review"]["checks"]
for requirement, result in initial_checks.items():
    print(f"{requirement}: {result['name']} -> {result['status']}")
assert len(initial_checks) == 7
assert all(result["passed"] for result in initial_checks.values())
assert report["checks"]["self_approval_refused"]
assert report["checks"]["independent_approval"]
assert report["checks"]["exact_approval_fingerprint"]
assert report["checks"]["expiring_approval"]
print("Maker self-approval REFUSED; exact independent local approval recorded")

## 4. Explain the template without claiming native enforcement

The local template's Conditional Access/access-package strings are **descriptive only**. Its conceptual default principles are not the tenant's real list of Microsoft default protections. No CA policy or `IT_Approved` attribute is assigned here. In a real deployment, templates refer to policies that already exist in Entra; referenced group/API permissions and the effective sign-in subject must be reviewed independently.

- Templates apply at **new activation**. Template edits do not change already-approved agents. This does not make referenced native policies immutable.
- **Report-only CA does not block access.** Begin with impact review; enabling enforcement requires separate authorization and is outside this lab.
- Agent-targeted CA requires the relevant agent-identity authentication flow. A developer credential and synthetic workshop roles are not the agent's own Entra identity.
- `IT_Approved` is an example custom attribute value, not automatic platform approval.
- **All agents** in a package request policy is request eligibility, not a resource grant to every agent.
- AI Administrator package capabilities do not imply authority to configure CA, assign custom attributes, or add arbitrary OAuth permissions/directory roles. The topic-specific package procedure requires Global Administrator for those privileged resource additions; custom attributes need explicit Attribute Assignment Administrator, including for a Global Administrator.

The next cell displays actual local receipt/package fields. The five-minute approval TTL and the example **180-day** package lifetime serve different purposes; 180 days is not a universal product default.

In [ ]:
receipt = report["publication_receipt"]
package = report["local_access_package"]
assert receipt["status"] == "DRY_RUN"
assert receipt["destination"] == "LOCAL_RECEIPT_ONLY"
assert package["authorization"] == "LOCAL_ONLY_NOT_ENTRA_ENTITLEMENT"
assert package["duration_days"] == 180
print(
    json.dumps(
        {
            "template": report["policy_template"]["name"],
            "conditional_access": report["policy_template"]["conditional_access"],
            "publication": receipt["status"],
            "package_kind": package["authorization"],
            "package_days": package["duration_days"],
            "reviewed_synthetic_resources": package["resources"],
        },
        indent=2,
    )
)

## 5. Compare blocked and permitted access

Before approval, the model has no local publication/authorization and denies both resources. After exact approval, it permits only synthetic read-only access. It separately attempts an out-of-package resource and a WRITE and must deny both.

The two abstract local resources stand in for reviewed entitlements. Real access packages can contain supported group memberships, directory roles or API permission grants; this model assigns none of them. A real package is also not a universal deny boundary: check direct grants, other packages, target-resource authorization and sign-ins before claiming **only through this package**.

In [ ]:
for before, after in zip(
    report["resource_access_before"], report["resource_access_after"], strict=True
):
    assert before["resource"] == after["resource"]
    assert before["status"] == "BLOCKED"
    assert after["status"] == "READ_ONLY"
    print(f"{before['resource']}: {before['status']} -> {after['status']}")
assert report["out_of_package_access"]["status"] == "BLOCKED"
assert report["write_access"]["status"] == "BLOCKED"
print("Outside-package resource: BLOCKED; WRITE: BLOCKED; native grants: NOT_EXECUTED")

## 6. Reject the unsafe update; retain v1

Imagine the second candidate arriving from a Copilot Studio maker after v1's Foundry organizational submission. Those platform labels are a conceptual mapping, not native requests created by this notebook. The local v2 deliberately removes owner/sponsor, fails the release gate, removes write approval, opens outbound access, raises clearance and requests an extra resource.

Exactly six **named** checks fail; its distinct synthetic agent identity still passes REQ02. Additional allowlist findings may exist, so the total findings list is not necessarily six entries. Approval and publication are refused. Confirm both the retained version number and an actual successful v1 local read; do not infer continued availability merely from a version label.

In [ ]:
failed = {key for key, result in report["update_review"]["checks"].items() if not result["passed"]}
assert failed == {"REQ01", "REQ03", "REQ04", "REQ05", "REQ06", "REQ07"}
assert report["update_review"]["checks"]["REQ02"]["passed"]
assert report["checks"]["failed_review_approval_refused"]
assert report["checks"]["rejected_update_publication_refused"]
assert report["current_version"] == 1
assert report["access_after_rejected_update"]["status"] == "READ_ONLY"
print("v2 rejected on named checks:", ", ".join(sorted(failed)))
print("v1 retained; v1 local read still READ_ONLY; native update/rejection: NOT_EXECUTED")

## 7. Prove what the local audit can and cannot prove

Verify the original audit against its captured head, then append an invalid record to a **copy** and verify rejection. Never alter the original. The head comes from this local report: retain it independently for a real evidence workflow. A local chain is not a remote immutable anchor, Purview ingestion, native admin approval or proof against an attacker rewriting both evidence and head.

Expected results: original True; altered copy False; original remains True.

In [ ]:
audit_path = Path(report["audit_path"])
head = report["audit_head"]
assert verify_audit(audit_path, head)
altered_copy = Path(workspace.name) / "altered-copy.audit.jsonl"
altered_copy.write_text(audit_path.read_text() + "{}\n")
assert not verify_audit(altered_copy, head)
assert verify_audit(audit_path, head)
print("Original audit verified; altered copy rejected; original preserved")

## 8. Optional read-only portal checklist

With separate read authorization, inspect only existing synthetic examples: **Agents > Settings > Templates** and **Agents > All agents > Requests**. Record privately whether the pages and Pending review/update/activate filters are visible; whether an existing custom template references an actual CA policy/package; and whether its reviewed audience, agent identity, resource roles and expiration match. Separately inspect the CA state and authenticated sign-in subject where permitted. Visibility is not enforcement.

**Do not** submit, approve, reject, publish, activate, block, save templates, assign packages/attributes, grant consent or change CA. Missing features, licensing, permissions or portal errors remain UNKNOWN/BLOCKED. The [full checklist and role matrix](../docs/participant-lab/request-approval-publish.md) explain the acceptance evidence. Never copy identifiers, customer information or raw portal captures into committed notebooks.

## Security explanation
Local publication and business-action approval are distinct from native administrative consent and entitlement assignment. Both remain synthetic here. Neither a source file, diagram animation, local PASS nor inventory entry establishes Agent 365/Defender/Purview activity. Do not blanket-label features GA or promise GCC/GCCH/DoD parity; verify each service and feature separately.

## Troubleshooting
Select the repository kernel if imports fail. Do not install packages inside cells. If assertions fail, inspect the actual retained report rather than changing expectations. The CLI intentionally rejects Azure lifecycle mode. If native pages are unavailable, continue locally and record the scoped native blocker.

## Cleanup
The final cell removes only this notebook's temporary exercise directory. Rerun the story cell to create a fresh run. Clear all outputs before sharing. CLI runs retain ignored state separately; do not delete broad directories or other people's evidence. No tenant cleanup is needed.

In [ ]:
exercise_directory = Path(workspace.name)
workspace.cleanup()
assert not exercise_directory.exists()
print("Scoped local exercise cleaned; no cloud state was changed")

## Source modules and official references

Implementation: [lifecycle.py](../src/foundry_security_accelerator/lifecycle.py), [audit.py](../src/foundry_security_accelerator/audit.py), [CLI](../src/foundry_security_accelerator/cli.py), [focused tests](../tests/test_lifecycle.py). Source code and local execution are not evidence of native integration.

- [Manage agent requests](https://learn.microsoft.com/en-us/microsoft-agent-365/admin/agent-requests): native states, review, publication wizard and pending updates.
- [Policy templates](https://learn.microsoft.com/en-us/microsoft-agent-365/admin/policy-template): defaults, existing Entra policies, administrator limitations and activation-only application.
- [Access packages for agent identities](https://learn.microsoft.com/en-us/entra/agent-id/agent-access-packages): resource roles, request scope, approval, expiration and privileged additions.
- [Conditional Access for autonomous agents](https://learn.microsoft.com/en-us/entra/identity/conditional-access/policy-autonomous-agents): actual agent sign-in, approval attributes, report-only impact review and licensing.
- [Foundry publication](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/publish-copilot): personal versus organizational scope, endpoint/version and Microsoft 365 data-flow review.
- [Copilot Studio Teams/Copilot distribution](https://learn.microsoft.com/en-us/microsoft-copilot-studio/publication-add-bot-to-microsoft-teams): maker publication and organizational approval are separate steps.
- [Foundry / Agent 365 integration](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/agent-365-integration): architecture context; prefer current topic-specific guidance for changing capabilities.

**Discuss:** Which evidence belongs to the maker, security/release reviewer, AI Administrator, identity-governance owner and CA/attribute administrators? What additional evidence would turn this local exercise into a separately authorized customer acceptance test?